# Reproducible mini-study: Internet use in Nigeria, 2010–2024

**Research question:** How did the percentage of Nigeria's population using the internet change between 2010 and 2024?

**Dataset:** World Bank World Development Indicators, indicator `IT.NET.USER.ZS` (Individuals using the Internet, % of population). Baseline snapshot compiled 11 November 2026.

**Observed baseline:** 11.50% in 2010 and 41.21% in 2024, a change of about 29.71 percentage points. These are reference values; the API cell below retrieves the live series again when run.

**Reproducibility:** Run cells from top to bottom in Google Colab or Jupyter with internet access. World Bank may revise historical values, so later runs may differ slightly. The included CSV preserves the baseline snapshot used for the finding and chart.


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

COUNTRY = 'NGA'
INDICATOR = 'IT.NET.USER.ZS'
START_YEAR = 2010
END_YEAR = 2024
DOWNLOAD_DATE = date.today().isoformat()

url = (
    f'https://api.worldbank.org/v2/country/{COUNTRY}/indicator/{INDICATOR}'
    f'?date={START_YEAR}:{END_YEAR}&format=json&per_page=100'
)
response = requests.get(url, timeout=30)
response.raise_for_status()
payload = response.json()
records = payload[1] if len(payload) > 1 and payload[1] else []
raw = pd.DataFrame(records)[['date', 'value']]
raw['year'] = pd.to_numeric(raw['date'], errors='coerce')
raw['internet_users_pct'] = pd.to_numeric(raw['value'], errors='coerce')
raw = raw[['year', 'internet_users_pct']].sort_values('year').reset_index(drop=True)
print('Download date:', DOWNLOAD_DATE)
print('Dataset URL:', url)
raw.head()

## Cleaning steps
1. Request only Nigeria (`NGA`), the selected indicator, and years 2010–2024.
2. Keep only the year (`date`) and reported observation (`value`) fields; rename them to descriptive column names.
3. Convert year and observation to numeric values; non-numeric values become missing (`NaN`).
4. Sort observations in ascending year order.
5. Do **not** remove missing-year rows silently. The analysis below excludes missing observations only when calculating the change and displays the number of available observations.
6. No other rows are deleted, no values are imputed, and no outliers are removed.


In [ ]:
clean = raw.copy()
valid = clean.dropna(subset=['internet_users_pct'])
print(f'Rows returned: {len(clean)}')
print(f'Non-missing observations: {len(valid)}')
print(f'Missing observations: {clean["internet_users_pct"].isna().sum()}')

if valid.empty:
    raise ValueError('No non-missing observations returned for the selected period.')
first = valid.iloc[0]
last = valid.iloc[-1]
absolute_change = last['internet_users_pct'] - first['internet_users_pct']
print(f'First available year: {int(first.year)} — {first.internet_users_pct:.2f}%')
print(f'Last available year: {int(last.year)} — {last.internet_users_pct:.2f}%')
print(f'Change: {absolute_change:.2f} percentage points')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(valid['year'], valid['internet_users_pct'], marker='o')
plt.title('Individuals using the Internet in Nigeria')
plt.xlabel('Year')
plt.ylabel('Internet users (% of population)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Calculation and interpretation
The change is calculated as: **last available non-missing value − first available non-missing value**, in percentage points. This is a difference in percentages, not a percentage growth rate.

## What this dataset cannot tell us
The indicator is a national population estimate. It does not identify which individuals use the internet, how often they use it, what they use it for, the quality or affordability of their connection, or differences by state, age, gender, disability, income, or rural/urban location. It also cannot establish why internet use changed or whether a particular policy caused the change. The latest year available may be earlier than the requested end year.

## Source and indicator definition
- World Bank indicator page: https://data.worldbank.org/indicator/IT.NET.USER.ZS?locations=NG
- API endpoint template: https://api.worldbank.org/v2/country/NGA/indicator/IT.NET.USER.ZS?date=2010:2024&format=json&per_page=100
- Indicator code: `IT.NET.USER.ZS`.


## Baseline snapshot
The included file `Nigeria_Internet_Use_2010_2024.csv` contains the 15 annual observations used for the dated chart and baseline finding. The API request above is the rerun route; compare its returned observations with the snapshot if the series has been revised.
